# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q('''
select t.title, a.name, a.country from artists a
left join tracks t on t.artist_id = a.artist_id
''')
#displays the title of the track, artist name and country after joining the id for the artist in the tracks and artists tables.

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Coastline,The Blue Ridge,US
3,Foothills,The Blue Ridge,US
4,Ridgeline,The Blue Ridge,US
5,Aurora,Kestrel,UK
6,Nightfall,Kestrel,UK
7,Untitled Demo,Kestrel,UK
8,Sol,Marisol,ES


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q('''
select t.genre, avg(t.seconds) as average from tracks t
group by t.genre
order by average desc
limit 1
''')
#grouped the genres and the average seconds in their songs, then ordered by descending and limited it to show one
#to see the genre with the longest average track length.

,genre,average
0,Electronic,287.5


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q('''
select p.user, count(*) as plays, count(distinct p.track_id) as distinct_plays from plays p
group by p.user
''')
#grouped plays by users, then displayed how many times they played a track and how many distinct tracks they listened to

,user,plays,distinct_plays
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q('''
select t.title, t.track_id from tracks t
left join plays p on t.track_id = p.track_id
where p.play_id is null
''')
#joined the track ids in the tracks and plays tables: where there is no match for the track_id in the plays tables,
#it means that nobody has played that track yet.

,title,track_id
0,Ridgeline,17
1,Untitled Demo,18


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q('''
select a.name, sum(t.seconds) as seconds, round(sum(t.seconds)/60.0, 1) as minutes from artists a
join tracks t on t.artist_id = a.artist_id
join plays p on p.track_id = t.track_id
group by a.name
order by seconds desc
''')
#grouped by artist name, then joined the track_ids and play_ids to create a table where each play is a row.
#Summing the seconds of each row while grouping by artist gives their total listening time.

,name,seconds,minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q('''
select t.track_id, t.title from tracks t
where genre is null
''')
#where genre != "pop" would have selected all tracks whose genre isn't pop
#written as is, we are displaying track and track_ids with no genre

,track_id,title
0,18,Untitled Demo


### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q('''
select p.played_on, count(*) as number_of_plays, count(distinct p.user) as users from plays p
group by p.played_on
order by p.played_on asc
''')
#groups based on date played, then counts how many time a track was played on that day and by how many distinct users

,played_on,number_of_plays,users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
q1 = q('''
select t.title, a.name, a.country from artists a
left join tracks t on t.artist_id = a.artist_id
''')
q4 = q('''
select t.title, t.track_id from tracks t
left join plays p on t.track_id = p.track_id
where p.play_id is null
''')
q3 = q('''
select p.user, count(*) as plays, count(distinct p.track_id) from plays p
group by p.user
''')

assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

I had the most trouble with question 5, as I wasn't sure at first how to count up the total seconds of the tracks played.  I didn't have a clear enough picture in my head to realize that after the joins, the query was keeping each play as a row.  Once I realized this, it made sense that sum(seconds) was sufficient to get my result.